In [1]:
from pathlib import Path
import sys

PROJ_ROOT = Path.cwd().parent

if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from features.engineering import add_features

from ml_challenge.config import INTERIM_DATA_DIR

In [3]:
train = pd.read_csv(INTERIM_DATA_DIR / "train_clean.csv")
test = pd.read_csv(INTERIM_DATA_DIR / "test_clean.csv")

print("Train:", train.shape)
print("Test :", test.shape)

Train: (699960, 26)
Test : (299984, 25)


In [4]:
TARGET = "is_fraud"
ID_COL = "id"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]

X_test = test.drop(columns=[ID_COL])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Test shape:", X_test.shape)

X shape: (699960, 24)
y shape: (699960,)
Test shape: (299984, 24)


In [5]:
X_fe = add_features(X)
X_test_fe = add_features(X_test)

print("Original train features:", X.shape[1])
print("Engineered train features:", X_fe.shape[1])

print("Original test features:", X_test.shape[1])
print("Engineered test features:", X_test_fe.shape[1])

Original train features: 24
Engineered train features: 29
Original test features: 24
Engineered test features: 29


In [6]:
new_features = [
    "red_flag_count",
    "amount_balance_ratio",
    "txn_rate_24h",
    "risk_velocity_interaction",
    "risk_flag_interaction"
]

print("New features:")
for feature in new_features:
    print("-", feature)

print("\nStatistics:")
print(X_fe[new_features].describe().T)

New features:
- red_flag_count
- amount_balance_ratio
- txn_rate_24h
- risk_velocity_interaction
- risk_flag_interaction

Statistics:
                              count        mean         std       min  \
red_flag_count             699960.0    0.235939    0.475296  0.000000   
amount_balance_ratio       699960.0    0.066704    0.205547  0.000028   
txn_rate_24h               699960.0    2.621698   17.161727  0.000000   
risk_velocity_interaction  699960.0  669.807879  479.899565  0.000000   
risk_flag_interaction      699960.0    8.423335   18.517358  0.000000   

                                  25%        50%         75%          max  
red_flag_count               0.000000    0.00000    0.000000     5.000000  
amount_balance_ratio         0.008791    0.01855    0.039609    14.609768  
txn_rate_24h                 0.261780    0.57971    1.388888   899.910009  
risk_velocity_interaction  332.142500  593.32000  923.650000  5766.390000  
risk_flag_interaction        0.000000    0.0000

In [7]:
print("Missing values:")
print(X_fe[new_features].isnull().sum())

print("\nInfinite values:")
print(np.isinf(X_fe[new_features]).sum())

Missing values:
red_flag_count               0
amount_balance_ratio         0
txn_rate_24h                 0
risk_velocity_interaction    0
risk_flag_interaction        0
dtype: int64

Infinite values:
red_flag_count               0
amount_balance_ratio         0
txn_rate_24h                 0
risk_velocity_interaction    0
risk_flag_interaction        0
dtype: int64


In [8]:
print(X_fe[new_features].dtypes)

red_flag_count                 int64
amount_balance_ratio         float64
txn_rate_24h                 float64
risk_velocity_interaction    float64
risk_flag_interaction        float64
dtype: object


In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_fe,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (559968, 29)
X_valid: (139992, 29)
y_train: (559968,)
y_valid: (139992,)


In [10]:
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

bool_cols = X_train.select_dtypes(
    include=["bool"]
).columns.tolist()

print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Boolean features:", len(bool_cols))

print("Numerical features:")
print(num_cols)

print("\nCategorical features:")
print(cat_cols)

print("\nBoolean features:")
print(bool_cols)

Numerical features: 18
Categorical features: 5
Boolean features: 6
Numerical features:
['amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'red_flag_count', 'amount_balance_ratio', 'txn_rate_24h', 'risk_velocity_interaction', 'risk_flag_interaction']

Categorical features:
['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

Boolean features:
['is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt']


In [11]:
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

In [12]:
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary"
    ))
])

In [13]:
bool_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary"
    ))
])

In [14]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("bool", bool_pipeline, bool_cols)
])

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test_fe)

In [16]:
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_valid shape:", X_valid_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

Processed X_train shape: (559968, 59)
Processed X_valid shape: (139992, 59)
Processed X_test shape : (299984, 59)


In [17]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for i, name in enumerate(feature_names):
    print(f"{i:02d}: {name}")

Number of processed features: 59
00: num__amount_usd
01: num__hours_since_last_txn
02: num__txn_count_last_24h
03: num__distance_from_home_km
04: num__card_age_months
05: num__customer_age
06: num__account_balance_usd
07: num__cvv_retry_count
08: num__velocity_score
09: num__time_of_day_hour
10: num__day_of_week
11: num__merchant_risk_score
12: num__prior_disputes
13: num__red_flag_count
14: num__amount_balance_ratio
15: num__txn_rate_24h
16: num__risk_velocity_interaction
17: num__risk_flag_interaction
18: cat__merchant_category_Crypto Exchange
19: cat__merchant_category_Electronics
20: cat__merchant_category_Fuel
21: cat__merchant_category_Gaming
22: cat__merchant_category_Gift Cards
23: cat__merchant_category_Groceries
24: cat__merchant_category_Healthcare
25: cat__merchant_category_Online Retail
26: cat__merchant_category_Restaurants
27: cat__merchant_category_Streaming
28: cat__merchant_category_Travel
29: cat__merchant_category_Utilities
30: cat__card_type_Amex
31: cat__card_type